# 🌍 Recreating Earth's Interior Structure (PREM Nested Model)

This notebook demonstrates how to create a 3D-printable model of Earth's layered interior (Section 3.2.2 of Koelemeijer & Winterbourne 2021). The model features multiple concentric nesting shells held together by magnets.

### 🌎 Scientific Context
Earth's interior is layered radially, mapped out by seismologists measuring the speed of seismic waves passing through the Earth. According to the **Preliminary Reference Earth Model (PREM)**:
- **Mantle**: Solid silicate rock, outer radius $6,371\text{ km}$, inner boundary (Core-Mantle Boundary) at $3,480\text{ km}$.
- **Outer Core**: Liquid iron-nickel, outer radius $3,480\text{ km}$, inner boundary at $1,221.5\text{ km}$.
- **Inner Core**: Solid iron-nickel, outer radius $1,221.5\text{ km}$.

We can model these layers to scale in millimeters:
1. **Mantle**: Outer radius 40 mm, inner cavity radius 21.8 mm (ratio: 0.546).
2. **Outer Core**: Outer radius 21.5 mm (with 0.3 mm clearance), inner cavity radius 7.5 mm (ratio: 0.35).
3. **Inner Core**: Solid sphere, radius 7.2 mm.

## Step 1: Import Libraries

In [ ]:
import os
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    calculate_displacement_scale
)

## Step 2: Build the Outer Mantle Shell

The Mantle shell features Earth's surface topography on the outside and is hollowed at the core boundary (21.8 mm).

In [ ]:
mantle_radius_mm = 40.0
cmb_ratio = 3480.0 / 6371.0  # 0.546

# 1. Initialize the hollow mantle model
mantle = GlobeModel(n_points=6000, radius=mantle_radius_mm, hollow=True, inner_ratio=cmb_ratio)

# 2. Load topography and displace outer surface (50x vertical exagg)
full_grid = GeographicGrid.from_netcdf(
    "../../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
grid_ds = GeographicGrid(
    lats=full_grid.lats[::15], lons=full_grid.lons[::15], grid=full_grid.grid[::15, ::15]
)
scale = calculate_displacement_scale(mantle_radius_mm, vertical_exagg=50.0, grid_units='m')
mantle.outer.displace(GridDisplacer(grid_ds), scale=scale)

# 3. Configure magnet cavities for the mantle mating ring
mantle.configure_magnets(
    diameter=5.0, height=2.0, n_magnets=3, position=0.0, add_bosses=True
)

# 4. Export hemispheres
os.makedirs('../../outputs', exist_ok=True)
mantle.export_hemispheres(
    "../../outputs/paper_mantle_top.stl",
    "../../outputs/paper_mantle_bottom.stl",
    engine='manifold'
)
print("Mantle hemispheres exported!")

## Step 3: Build the Outer Core Shell

The Outer Core has a slightly smaller outer radius (21.5 mm) to slide easily into the Mantle's cavity, and is hollowed at the Inner Core boundary (7.5 mm).

In [ ]:
outer_core_radius_mm = 21.5
icb_ratio = 1221.5 / 3480.0  # 0.35

# 1. Initialize the hollow outer core model (no topography needed on core surface)
outer_core = GlobeModel(n_points=3000, radius=outer_core_radius_mm, hollow=True, inner_ratio=icb_ratio)

# 2. Configure smaller magnet cavities for core joint alignment
outer_core.configure_magnets(
    diameter=3.0, height=1.5, n_magnets=3, position=0.0, add_bosses=True
)

# 3. Export core hemispheres
outer_core.export_hemispheres(
    "../../outputs/paper_outer_core_top.stl",
    "../../outputs/paper_outer_core_bottom.stl",
    engine='manifold'
)
print("Outer core hemispheres exported!")

## Step 4: Build the Solid Inner Core Sphere

The Inner Core is a solid sphere of radius 7.2 mm.

In [ ]:
inner_core_radius_mm = 7.2

# Create solid core sphere
inner_core = GlobeModel(n_points=1000, radius=inner_core_radius_mm, hollow=False)
inner_core.export("../../outputs/paper_inner_core.stl")
print("Solid Inner Core exported!")